
# summary by far:

| Model Type               | Speed     | MAP\@10  | Description / Best Use                                    |
| ------------------------ | --------- | -------- | --------------------------------------------------------- |
| **TF-IDF**               | Very Fast | 0.29     | Simple lexical baseline (name + description)              |
| **BM25**                 | Fast      | 0.34     | Stronger lexical match (no cleaning performs better here) |
| **SBERT (bi-encoder)**   | Moderate  | 0.36     | Semantic search with precomputed embeddings               |
| **BM25 + Cross-Encoder** | Slow      | **0.42** | **Best reranking accuracy** on top 50 candidates          |


**steps taken:** 
The initial retrieval (BM25) is strong enough to surface plausible candidates.

The cross-encoder is doing its job — applying nuanced pairwise scoring that bi-encoders and TF-IDF simply can't.

**why map@10 is hard to improve**

- label mismatch: I checked that the 'Exact' match has min of 1 product, 25% at 4 product, and 50% at 28 products, even with 28 matches, the top10 may not include all of them.
 
- my model may return great semantic matches (sentence BERT) that aren't labeled as 'Exact', and those own't get any MAP credits

- WANDS Labeling Policy

Wayfair (in WANDS) labels products as "Exact" only if they're identical or near-identical to the target product in a browsing session. It’s not recall-complete for all plausible matches. 

**why not hybrid scoring using BM25 + SBERT Fusion:**

this is worth trying, but can most likely improve the score by 0.1

**why ot Augment Text with Select Metadata: like the product features, product class, category hierarchy**

I checked the search query length in the query_df, the query itself is relatively very short, with mean and median at 3 words, and maximum at 10. My thoughts were that by improving the context in the product side alone, will have relatively minimal impact on the match result. 


**what's next**

I want to reach a higher MAP@K score (close to 0.6) taht would indicates on average, each query retrieves a significant number of truly relevant products (often labeled “Exact”) in the top 10 results, and does so with high ranking precision.

| Step                                            | Description | Est. MAP\@10 Gain |
| ----------------------------------------------- | ----------- | ----------------- |
| Better cross-encoder (e.g., MiniLM-L12-v2)      | +0.03–0.05  |                   |
| Rerank top 100+ instead of 50                   | +0.01–0.03  |                   |
| Fine-tune cross-encoder or bi-encoder           | +0.05–0.10  |                   |
| Hybrid candidate retrieval (BM25 + SBERT)       | +0.01–0.03  |                   |
| Structured filtering/boosting (category, brand) | +0.01–0.03  |                   |

**why I hesitate on trying LLM APIs**
- LLM isn't commonly used for (or at least not directly) for reranking yet.
- large number of queries can slow things down, not ideal on production envrionment,
- it lacks built-in ranking scores, its a generalist, so i need to do promopt engineer to do
'Given query Q and product P, rate relevance on a 1–5 scale',
- whereas Models like cross-encoder/ms-marco-MiniLM or deberta-v3 are purpose-built for dense pairwise ranking.

In [ ]:
# %pip install -U sentence-transformers

In [3]:
import sys
import os
import json 
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

ENV = 'dev'

# Load config
with open("config.json") as f:
    config = json.load(f)

if ENV == 'dev':
    base_path = config[f"{ENV}_path"]  
    data_path = os.path.join(base_path, "data")
    model_path = os.path.join(base_path, "models")
    print("Base path:", base_path)    
    print("Data path:", data_path)
    print("Model path:", model_path)

Base path: /Users/jillchow/HBS/hbs_search_engine
Data path: /Users/jillchow/HBS/hbs_search_engine/data
Model path: /Users/jillchow/HBS/hbs_search_engine/models


In [4]:
queryfile_name = "query.csv" 
queryfile_path = os.path.join(data_path, queryfile_name)
productfile_name = "product.csv" 
productfile_path = os.path.join(data_path, productfile_name)
labelfile_name = "label.csv" 
labelfile_path = os.path.join(data_path, labelfile_name)
query_df = pd.read_csv(queryfile_path, sep='\t')
product_df = pd.read_csv(productfile_path, sep='\t')
label_df = pd.read_csv(labelfile_path, sep='\t')


In [ ]:
from sentence_transformers import CrossEncoder
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')  


/Users/jillchow/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [5]:
from rank_bm25 import BM25Okapi
import nltk
from nltk.tokenize import word_tokenize

# nltk.download('punkt')

corpus = (product_df['product_name'] + ' ' + product_df['product_description']).fillna("").astype(str).tolist()
tokenized_corpus = [word_tokenize(doc.lower()) for doc in corpus]
bm25 = BM25Okapi(tokenized_corpus)


def get_top_products_bm25(query, top_n=10):
    tokenized_query = word_tokenize(query.lower())
    scores = bm25.get_scores(tokenized_query)
    top_indices = np.argsort(scores)[-top_n:][::-1]
    return top_indices

In [ ]:
def get_candidates(query, product_df, top_n=50):
    indices = get_top_products_bm25(query, top_n=top_n)
    candidates = product_df.iloc[indices]
    return candidates, indices

In [11]:
def rerank_with_cross_encoder(query, candidates, indices, top_n=10):
    product_texts = (
        candidates['product_name'] + ' ' +
        candidates['product_description']
    ).fillna('').tolist()
    
    query_product_pairs = [(query, doc) for doc in product_texts]
    scores = cross_encoder.predict(query_product_pairs)

    # Get top indices
    reranked = np.argsort(scores)[::-1][:top_n]
    top_indices = [indices[i] for i in reranked]
    return top_indices


In [20]:
def get_top_products_cross_encoder(query, product_df, top_k=10, top_n = 50):
    candidates, indices = get_candidates(query, product_df, top_n=top_n)
    top_indices = rerank_with_cross_encoder(query, candidates, indices, top_n=top_k)
    return top_indices


In [21]:
# query_df['top_product_ids'] = query_df['query'].apply(
#     lambda q: product_df.iloc[get_top_products_cross_encoder(q, product_df, top_k= 10)].product_id.tolist()
# )
# 
def run_cross_encoder_retrieval(query_df, product_df, top_k=10):
    query_df['top_product_ids'] = query_df['query'].apply(
        lambda q: product_df.iloc[get_top_products_cross_encoder(q, product_df, top_k=top_k, top_n = 100)].product_id.tolist()
    )
    return query_df

query_df = run_cross_encoder_retrieval(query_df, product_df, top_k=10)


In [22]:
# update to use the label_df, and add to the parameters
def get_exact_matches_for_query(query_id, label_df):
    grouped_label_df = label_df.groupby('query_id')
    query_group = grouped_label_df.get_group(query_id)
    exact_matches = query_group.loc[query_group['label'] == 'Exact']['product_id'].values
    return exact_matches

# adding the list of exact match product_IDs from labels_df

query_df['relevant_ids'] = query_df['query_id'].apply(
      lambda qid: get_exact_matches_for_query(qid, label_df)
)



query_df.head(10)

,query_id,query,query_class,top_product_ids,relevant_ids,map@k
0,0,salon chair,Massage Chairs,"[7467, 7465, 7468, 25431, 7466, 24010, 42329, ...","[25434, 42931, 2636, 42923, 41156, 5936, 22390...",0.575000
1,1,smart coffee table,Coffee & Cocktail Tables,"[33698, 21580, 1308, 36315, 20331, 19898, 2267...","[9929, 5235, 37304, 25973, 16679, 29449, 33698...",0.125000
2,2,dinosaur,Kids Wall Décor,"[24094, 12130, 34737, 10552, 10553, 3754, 2104...","[4205, 4202, 4204, 36622, 29777, 40289, 10539,...",1.000000
3,3,turquoise pillows,Accent Pillows,"[5998, 21244, 23599, 19100, 6874, 24116, 13577...","[18909, 12386, 12436, 32704, 12201, 18293, 404...",0.380000
4,4,chair and a half recliner,Recliners,"[22576, 2656, 14432, 1526, 12383, 6458, 8198, ...","[5488, 6098, 42393, 16598, 41662, 40331, 24881...",0.000000
5,5,sofa with ottoman,Sectionals,"[20646, 9686, 33252, 23610, 33253, 33251, 8497...","[33253, 19757, 26053, 6986, 25713, 14091, 4139...",1.000000
6,6,acrylic clear chair,Dining Chairs,"[33512, 21914, 5298, 25143, 18662, 19248, 2955...","[41828, 32573, 25711, 1454, 28922, 25147, 1981...",0.671111
7,7,driftwood mirror,Wall & Accent Mirrors,"[34083, 34926, 11297, 15852, 3647, 37655, 1737...","[37651, 19768, 27648, 37652, 37645, 34787, 142...",0.815437
8,8,home sweet home sign,Wall Décor,"[19132, 30557, 30562, 28089, 8990, 8778, 21994...","[30082, 21994, 19132, 23011, 22077, 889, 8301,...",0.790437
9,9,coffee table fire pit,Outdoor Fireplaces,"[7996, 7997, 27916, 7509, 13615, 8715, 6797, 2...","[20907, 30092, 3288, 29691, 30826, 13615, 2936...",1.000000


In [23]:
import importlib
import helper
importlib.reload(helper)
from helper import calculate_tfidf, get_top_products, map_at_k, get_top_product_ids_for_query, get_exact_matches_for_query

In [18]:
query_df['map@k'] = query_df.apply(lambda x: map_at_k(x['relevant_ids'], x['top_product_ids'], k=10), axis=1)
print("🧼 BM25 + cross-encoder rerank MAP@10:", query_df['map@k'].mean())

🧼 BM25 + cross-encoder rerank MAP@10: 0.42336291513133034


In [ ]:
query_df['map@k'] = query_df.apply(lambda x: map_at_k(x['relevant_ids'], x['top_product_ids'], k=10), axis=1)
print("🧼 BM25(top 100) + cross-encoder rerank MAP@10:", query_df['map@k'].mean())


🧼 BM25(top 100) + cross-encoder rerank MAP@10: 0.4284074831900353
